This example shows how one can take AA data generated using Gromacs or LAMMPS, apply a CG mapping to it, calculate 
its RDF, and save the data in a format that is ready to feed to the CG model training script.

In [ ]:
import os
import itertools

import numpy as np
import matplotlib.pyplot as plt

from hippynn.molecular_dynamics.misc import SpeciesLookup
from hippynn.molecular_dynamics.rdf import calculate_rdf
from hippynn.molecular_dynamics.readers import extract_trajectory_data
from hippynn.molecular_dynamics.coarse_grain import coarse_grain_all, cg_one_center_of_mass_pbc, cg_one_sum, cg_one_mass_weighted_average
from hippynn.molecular_dynamics.writers import write_extxyz

In [ ]:
results_folder = "aa_to_cg_workflow_results"
os.makedirs(results_folder, exist_ok=True)

In [ ]:
# We can load Gromacs outputs using a .trr file and a topology file
# NOTE: You will need to supply these files for your own trajectory, as they are not provided
trajectory = "ec.trr"
topology = "system.pdb" # alternatively .gro file
aa_data = extract_trajectory_data(trajectory=trajectory, topology=topology)

for key, value in aa_data.items():
    print(f"{key}, {value.shape}")

In [ ]:
# Or we can load LAMMPS outputs using a .lammpstrj file and a topology file
# NOTE: You will need to supply these files for your own trajectory, as they are not provided
trajectory = "output.lammpstrj"
topology = "system.data"

# The files I used here were generated using a force field that has its own system to assigning a 'type' to each atom. This will tell the reader how
# to translate between these types and the atomic number/symbol
type_correction_dict = {
    '4': "C",
    '19': "H",
    '27': "H",
    '31': "H",
    '33': "Cl",
    '40': "N",
    '58': "O",
}

aa_data = extract_trajectory_data(trajectory=trajectory, topology=topology, type_correction_dict=type_correction_dict, start=0, stop=400, stride=4)

for key, value in aa_data.items():
    print(f"{key}, {value.shape}")

In [ ]:
aa_positions = aa_data["positions"]
aa_velocities = aa_data["velocities"]
aa_forces = aa_data["forces"]
aa_cells = aa_data["cells"]
aa_masses = aa_data["masses"]
aa_species_numbers = aa_data["species_numbers"]
aa_species_symbols = aa_data["species_symbols"]
aa_mol_ids = aa_data["mol_ids"]

**NOTE**: There are no unit conversions being performed in this example, so the training data and thus the model will use the units that were used for the AA simulation data. If this is not what you want, I suggest to convert your units before proceeding with the steps below.

In [ ]:
# Next we will apply the coarse-graining mappings. See hippynn/hippynn/molecular_dynamics/coarse_grain.py for more information about how we implement this.
# You can create your own coarse-grained mappings and use this method to apply them to the data.
# 
# We need:
#   - a function `coarse_grain_one` which determines how one coarse-grained quantity is computed on a single bead 
#   - an array `atom_to_bead` which assigns a bead ID to each atom 
# Here, we'll use:
#   - the pre-defined `cg_one_center_of_mass_pbc` bead mapping function which uses a center-of-mass mapping for positions with PBC
#   - the molecule IDs since we are mapping each molecule to one bead

cg_positions = coarse_grain_all(
    aa_positions, 
    atom_to_bead=aa_mol_ids,
    coarse_grain_one=cg_one_center_of_mass_pbc,
    masses=aa_masses, 
    cells=aa_cells
)

print(cg_positions.shape)

# NOTE: These functions were designed with the goals of flexibility and ease-of-use. As a result, they are not
# optimally fast. If this is an issue for your workflow, there should be ways to speed up these operations, 
# such as using a JIT compiler

In [ ]:
# For forces, we map the forces on each atom in a molecule to the sum of the forces
# Since `cg_one_sum` does not require mass or cell data, we do not need to provide it to `coarse_grain_all`
cg_forces = coarse_grain_all(
    aa_forces, 
    atom_to_bead=aa_mol_ids,
    coarse_grain_one=cg_one_sum,
)

print(cg_forces.shape)

In [ ]:
# For masses, again we apply the sum function. Since the masses are of shape (n_atoms,) rather than
# (n_frames, n_atoms,), we set `single_frame=True` so that `coarse_grain_all` does not try to iterate 
# over a non-existant 'frames' axis.

cg_masses = coarse_grain_all(
    aa_masses,
    atom_to_bead=aa_mol_ids,
    coarse_grain_one=cg_one_sum,
)

print(cg_masses.shape)

In [ ]:
# For velocities, we use a mass-weighted average

cg_velocities = coarse_grain_all(
    aa_velocities,
    atom_to_bead=aa_mol_ids,
    coarse_grain_one=cg_one_mass_weighted_average,
    masses=aa_masses,
)

print(cg_velocities.shape)

In [ ]:
# We'll define a new coarse_grain_one function to map the sets of atom species to a bead species
# You might need to do this different depending on how your beads are determined
# In some instances, it might be easier to create the cg_species matrix by hand
def cg_one_species_numbers(bead_values):
    # all beads in this example with this atom list are choline molecules
    if sorted(list(bead_values)) == [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6, 6, 6, 6, 6, 7, 8]:
        return 1
    # all beads in this example with this atom list are chloride
    if sorted(list(bead_values)) == [17]:
        return 2
    # all beads in this example with this atom list are ethylene glycol molecules
    if sorted(list(bead_values)) == [1, 1, 1, 1, 1, 1, 6, 6, 8, 8]:
        return 3
    
cg_species = coarse_grain_all(
    aa_species_numbers,
    atom_to_bead=aa_mol_ids,
    coarse_grain_one=cg_one_species_numbers,
)

print(cg_species.shape)

In [ ]:
# We can do almost the same thing to get the bead names
# We could also just do this with a list comprehension
def cg_one_species_names(bead_values):
    # all beads in this example with this atom list are choline molecules
    if sorted(list(bead_values)) == [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6, 6, 6, 6, 6, 7, 8]:
        return "Cho"
    # all beads in this example with this atom list are chloride
    if sorted(list(bead_values)) == [17]:
        return "Cl"
    # all beads in this example with this atom list are ethylene glycol molecules
    if sorted(list(bead_values)) == [1, 1, 1, 1, 1, 1, 6, 6, 8, 8]:
        return "EG"
    
cg_species_names = coarse_grain_all(
    aa_species_numbers,
    atom_to_bead=aa_mol_ids,
    coarse_grain_one=cg_one_species_names,
)

print(cg_species_names.shape)

In [ ]:
# Finally, we'll calculate the RDF using the CG positions
bins, rdf = calculate_rdf(
    positions=cg_positions,
    cutoff=15,
    cells=aa_cells, # AA and CG cells are the same
)

In [ ]:
# this will look a bit odd if it combines interactions between mutiple species of beads
fig, ax = plt.subplots()
ax.plot(bins, rdf);

In [ ]:
# We can also calculate RDFs for each species pair

bins, rdf, species_rdfs = calculate_rdf(
    positions=cg_positions,
    cutoff=15,
    cells=aa_cells, # AA and CG cells are the same
    species=cg_species,
)

In [ ]:
cg_species_lookup = SpeciesLookup(cg_species, cg_species_names)

unique_species = np.unique(cg_species)

fig, ax = plt.subplots()

for sp1, sp2 in itertools.combinations_with_replacement(unique_species, 2):
    sp1_name = cg_species_lookup.number_to_name(sp1)
    sp2_name = cg_species_lookup.number_to_name(sp2)
    ax.plot(bins, species_rdfs[f"rdf_values_{sp1}-{sp2}"], label = f"{sp1_name}-{sp2_name}")

ax.legend();

In [ ]:
# This will create the training data file that we can provide to hippynn to train a CG model

values = {
    "positions": cg_positions,
    "velocities": cg_velocities, 
    "forces": cg_forces,
    "cells": aa_cells, # AA and CG cells are the same
    "species": cg_species, 
    "masses": cg_masses,
    "rdf_bins": bins, 
    "rdf_values": rdf,
    "species_names": cg_species_names, # need to run CG model in LAMMPS
}

values.update(species_rdfs) # needed for species pair specific repulsive potential

np.savez(os.path.join(results_folder, "training_data.npz"), **values)

In [ ]:
# Additionally, we can write the CG (or AA) data to a .extxyz file to view in Ovito or other visualization software

write_extxyz(filename=os.path.join(results_folder, "aa_beads_traj.extxyz"), positions = cg_positions, species=cg_species, cells=aa_cells, velocities=cg_velocities, forces=cg_forces)
write_extxyz(filename=os.path.join(results_folder, "aa_traj.extxyz"), positions = aa_positions, species=aa_species_symbols, cells=aa_cells, velocities=aa_velocities, forces=aa_forces)